In [3]:
import pandas as pd
from pathlib import Path
from IPython.display import display

# Load WhatsApp chat exports
Reads all .txt files from the data/raw folder and loads them into a pandas DataFrame with columns: filename, path, text, and lines.

In [168]:
from pathlib import Path
from IPython.display import display

data_dir = Path('..') / 'data' / 'raw'
files = sorted(data_dir.glob('*.txt'))
rows = []

for p in files:
    try:
        text = p.read_text(encoding='utf-8')
    except UnicodeDecodeError:
        text = p.read_text(encoding='latin-1')
    lines = text.splitlines()
    rows.extend(lines[1:])

df_chats = pd.DataFrame({'text': rows})
print(f'Loaded {len(df_chats)} lines from {len(files)} chat files in {data_dir.resolve()}')
display(df_chats.head())


Loaded 48922 lines from 18 chat files in C:\Users\arsal\Desktop\second-voice\data\raw


,text
0,"4/11/25, 2:40 AM - +92 318 0067351: aoa yaar m..."
1,"4/11/25, 2:40 AM - +92 318 0067351: yaad se"
2,"4/11/25, 8:03 AM - ~Arsalan Qasim: Sahi hy.. M..."
3,"4/11/25, 8:04 AM - +92 318 0067351: Ok"
4,"4/11/25, 8:04 AM - +92 318 0067351: M 8 30.pr ..."


In [169]:
import re

# Extract leading WhatsApp timestamps like "4/11/25, 2:40\u202fAM - " into a timestamp column
timestamp_pattern = r'^\s*(\d{1,2}/\d{1,2}/\d{2},\s*\d{1,2}:\d{2}\s*(?:AM|PM|am|pm|\u202fAM|\u202fPM)?)\s*-\s*'
df_chats['timestamp'] = df_chats['text'].astype(str).str.extract(timestamp_pattern, expand=False)
df_chats['text'] = df_chats['text'].astype(str).str.replace(timestamp_pattern, '', regex=True)

# Extract sender names like "~Arsalan Qasim:" or "+92 318 0067351:" into a person column
person_pattern = r'^\s*(?:~)?([^:]+?):\s*'
df_chats['person'] = df_chats['text'].str.extract(person_pattern, expand=False).str.strip()
df_chats['text'] = df_chats['text'].str.replace(person_pattern, '', regex=True)

display(df_chats[['timestamp', 'person', 'text']].head())


,timestamp,person,text
0,"4/11/25, 2:40 AM",+92 318 0067351,aoa yaar main na tumhare sath DE repeat kr rha...
1,"4/11/25, 2:40 AM",+92 318 0067351,yaad se
2,"4/11/25, 8:03 AM",Arsalan Qasim,Sahi hy.. Main university pounch kar tum sy ly...
3,"4/11/25, 8:04 AM",+92 318 0067351,Ok
4,"4/11/25, 8:04 AM",+92 318 0067351,M 8 30.pr niklu ga yha se agr tune phle Jana h...


In [170]:
# Remove rows where the message text is just a media placeholder
df_chats = df_chats[~df_chats['text'].astype(str).str.contains('<Media omitted>', na=False)].copy()

In [171]:
# Remove rows where person is missing
df_chats = df_chats[df_chats['person'].notna()].copy()
print(f"Removed rows with missing person. Remaining rows: {len(df_chats)}")

Removed rows with missing person. Remaining rows: 32106


In [172]:
# Remove rows where message text is empty or only whitespace
mask = df_chats['text'].notna() & (df_chats['text'].astype(str).str.strip() != '')
df_chats = df_chats[mask].copy()

In [173]:
import pandas as pd

# 1. Convert timestamp column to datetime objects
df_chats["timestamp"] = pd.to_datetime(
    df_chats["timestamp"], format="%m/%d/%y, %I:%M %p"
)

# 2. Identify consecutive blocks (sender changes OR time gap >= 30 mins)
sender_changed = df_chats["person"] != df_chats["person"].shift()
time_gap_large = df_chats["timestamp"].diff() >= pd.Timedelta(minutes=30)

# 3. Apply changes directly back to df_chats
df_chats = (
    df_chats.groupby((sender_changed | time_gap_large).cumsum())
    .agg(
        {
            "person": "first",
            "timestamp": "last",  # Take the timestamp of the last message in the burst
            "text": lambda x: " ".join(x.astype(str)),  # Combine the text strings
        }
    )
    .reset_index(drop=True)
)


In [174]:
# 1. Create temporary columns looking at the immediate next row
df_chats["next_person"] = df_chats["person"].shift(-1)
df_chats["next_text"] = df_chats["text"].shift(-1)

# 2. Apply the logic: Keep rows where an outsider speaks and Arsalan replies next
final_pairs = df_chats[
    (df_chats["person"] != "Arsalan Qasim")
    & (df_chats["next_person"] == "Arsalan Qasim")
].copy()

# 3. Clean up and rename columns, keeping the contact's identity
final_pairs = final_pairs.rename(
    columns={
        "person": "contact_person",  # The person you are replying to
        "text": "input_text",
        "next_text": "reply_text",
    }
)

# 4. Filter down to the final required columns
df_chats = final_pairs[
    ["contact_person", "input_text", "reply_text"]
].reset_index(drop=True)


In [175]:
import numpy as np

start = np.random.randint(0, len(df_chats) - 10 + 1)

df_chats.iloc[start:start + 10]

,contact_person,input_text,reply_text
1114,Alisha,Is main serf male apply kar sakty hn ya female...,Sirf male https://joinpakarmy.gov.pk/ Tum army...
1115,Alisha,Lahore jao ge Kya Tum,Han
1116,Alisha,Kips ki serf biology prep book chahiha thi mhu...,Sahi hy ly aho ga Par main late aho ga
1117,Alisha,Bus book nhi Lena main ne Quetta se leli,Sahi
1118,Alisha,Pkg karo Mera,Habt Han
1119,Alisha,Serf Whatsapp ka karo,Konsa Monthly?
1120,Alisha,Hao,Raat ko Karo ga Abhi nahi ho raha
1121,Alisha,Submit nhi horaha,password bhejo ??
1122,Alisha,$zP6#R?ACJ_d6Qs Album main jao phir pic upload...,cnic back
1123,Alisha,Ho gaya?? Ho gaya ???,ya details check karo sahi han ???


In [ ]:
X = f"[PERSON={df_chats['contact_person']}] {df_chats['input_text']}"
y = df_chats['reply_text']